# Multigrain Tutorial

In [1]:
import numpy as np
import pyvista as pv

import tommos_adapter as ta

pv.set_jupyter_backend("trame")

## Generate a multigrain mesh using tommos and neper

In [2]:
!tommos mesh \
    --geom poly_gb \
    --extent 40,40,40 \
    --h 3 \
    --n 4 \
    --id 7 \
    --gb-thickness 2 \
    --gb-h 1.5 \
    --out-name multigrain


========================    N   e   p   e   r    =======================
Info   : A software package for polycrystal generation and meshing.
Info   : Version 5.0.0
Info   : Built with: gsl|muparser|opengjk|openmp|nlopt|libscotch (full)
Info   : Running on 10 threads.
Info   : <https://neper.info>
Info   : Copyright (C) 2003-2026, and GNU GPL'd, by Romain Quey, CNRS.
Info   : No initialization file found (`/Users/michael/.neperrc').
Info   : ---------------------------------------------------------------
Info   : MODULE  -T loaded with arguments:
Info   : [ini file] (none)
Info   : [com line] -n 4 -id 7 -morpho gg -morphooptistop val=0.1 -domain
         cube(40.0,40.0,40.0):translate(0,0,0) -format tess,obj -reg 1
Info   : ---------------------------------------------------------------
Info   : Reading input data...
Info   : Creating domain...
Info   : Creating tessellation...
Info   :   - Setting seeds... 255075100%
Info   :   - Running tessellation...
Info   :     > Initial solution

## Inspect the generated mesh using pyvista

### Import the vtu-file using pyvista and check the regions

The following cell imports the generated mesh using pyvista. The number of cells and points, the outer extends in x, y, z directions, the material IDs and related number of tetrahedra is displayed. 
This provides a compact overview of the meta data of the mesh.

In [3]:
mesh_nm = pv.read("multigrain.vtu")
material_ids = np.asarray(mesh_nm.cell_data["mat_id"])

print(mesh_nm)
print("Material IDs:", np.unique(material_ids))

ids, counts = np.unique(material_ids, return_counts=True)
for material_id, count in zip(ids, counts, strict=True):
    print(f"Material {material_id}: {count} tetrahedra")

UnstructuredGrid (0x3158e1d20)
  N Cells:    416518
  N Points:   82932
  X Bounds:   -2.000e+01, 2.000e+01
  Y Bounds:   -2.000e+01, 2.000e+01
  Z Bounds:   -2.000e+01, 2.000e+01
  N Arrays:   1
Material IDs: [1 2 3 4 5]
Material 1: 16620 tetrahedra
Material 2: 34335 tetrahedra
Material 3: 66248 tetrahedra
Material 4: 16837 tetrahedra
Material 5: 282478 tetrahedra


### Plot two-dimensional slices of the tetrahedron mesh using pyvista

In the following the xy-slice of the thedrahedral mesh is displayed. Use the slider for selecting the z-coordinate of the xy-slice.

In [4]:
material_ids = np.asarray(mesh_nm.cell_data["mat_id"])
z_min, z_max = mesh_nm.bounds[4], mesh_nm.bounds[5]
z_initial = 0.0

section = mesh_nm.slice(
    normal=(0, 0, 1),
    origin=(0, 0, z_initial),
)

slice_plotter = pv.Plotter()


slice_plotter.add_mesh(
    mesh_nm.outline(),
    color="black",
    line_width=2,
)


slice_plotter.add_mesh(
    section,
    scalars="mat_id",
    categories=True,
    cmap="tab10",
    clim=(material_ids.min(), material_ids.max()),
    show_edges=True,
    scalar_bar_args={"title": "Region"},
)


def update_slice(
    z,
    source_mesh=mesh_nm,
    target_section=section,
    target_plotter=slice_plotter,
):
    """Move the section along the z-axis."""
    new_section = source_mesh.slice(
        normal=(0, 0, 1),
        origin=(0, 0, float(z)),
    )
    target_section.copy_from(new_section, deep=True)
    target_plotter.render()


slice_plotter.add_slider_widget(
    update_slice,
    rng=(z_min, z_max),
    value=z_initial,
    title="z [nm]",
    fmt="%.1f",
    interaction_event="end",
)

slice_plotter.show(jupyter_backend="trame")

In [5]:
gb_id = material_ids.max()

grains = mesh_nm.extract_cells(material_ids != gb_id)
grain_boundary = mesh_nm.extract_cells(material_ids == gb_id)

overview_plotter = pv.Plotter()
overview_plotter.add_mesh(
    grains,
    scalars="mat_id",
    categories=True,
    cmap="tab10",
    show_edges=False,
)
overview_plotter.add_mesh(
    grain_boundary,
    color="lightgray",
    opacity=0.12,
)
overview_plotter.add_axes()
overview_plotter.show(jupyter_backend="trame")

In [6]:
mesh_si = mesh_nm.copy(deep=True)
mesh_si.points *= ta.scripts.DEFAULT_MESH_UNIT

region_id = mesh_si.get_array("mat_id").astype(int)
mesh_si.cell_data["region_id"] = region_id
mesh_si.cell_data.remove("mat_id")

ids = np.unique(region_id)
gb_id = ids.max()
names = [f"grain_{i}" if i != gb_id else "grain_boundary" for i in ids]

mesh_si.field_data["region_ids"] = ids
mesh_si.field_data["region_names"] = names
mesh_si.point_data["m"] = np.tile((0, 0, 1), (mesh_si.n_points, 1))